# RFSoC 4×2 · Method B — TX
A 板專用。先啟動 B 板 RX，待原硬體提示 `RX_WAITING_FOR_PILOT` 後啟動 TX。
部署完整的 `A_TX/` 並重啟 kernel。只有一塊板時，請開啟 `B_RX/` 的 notebook，以預設 `single_board` 模式驗證 TX 和 RX 的 BRAM 資料路徑。
保留 RFJ2、ACK、BRAM、EOS 與相同 FPGA overlay。原生預覽每個 block 取最後一張 JPEG。

## 1. TX configuration

In [ ]:
from pathlib import Path
import json, hashlib, re
from collections import deque, OrderedDict
import threading
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from webcam_live import run_live
from webcam_stream import WIDTH, HEIGHT, JPEG_BUDGET
from webcam_latency import show_latency

BUNDLE = Path.cwd()
BITFILE = BUNDLE / "ofdm_qam16.bit"
OUTPUT_ROOT = BUNDLE / "received_live_webcam"
FPS = 30.0
AMPLITUDE = 4096
LIVE_PREVIEW = True
RECORDING_CODEC = "auto"
PAIR_SESSION = "test_001"  # paired export only; use a NEW matching ID on both boards
assert re.fullmatch(r"[A-Za-z0-9_-]+", PAIR_SESSION)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ROLE = "send"
CAMERA = 0  # 或有限長度影片路徑
DURATION_S = 30.0  # None = 手動 Interrupt / 影片 EOF；不含初始化
JPEG_QUALITY = (85, 20, 95, 5)  # start, min, max, step
MEASURE_LOCAL_PSNR = True
EXPORT_REFERENCE = False  # 配對 PSNR 才開啟；同步磁碟 I/O 會影響效能
MAX_LOG_FRAMES = 1000
print("JPEG target budget:", JPEG_BUDGET, "bytes")

## 2. TX 原始影像與 JPEG 分析
callback 由 Capture 執行緒在正式 JPEG 編碼後呼叫；不改封包格式。
記錄的是成功編碼、可裝入 block 的影像，仍可能被 backlog 淘汰，並非全部已送達。
本地 PSNR 需要額外解碼；若只量效能，關閉 `MEASURE_LOCAL_PSNR`。

In [ ]:
tx_log = deque(maxlen=MAX_LOG_FRAMES)
tx_errors = deque(maxlen=20)
reference_dir = OUTPUT_ROOT / "pair_reference" / PAIR_SESSION
if EXPORT_REFERENCE:
    reference_dir.mkdir(parents=True, exist_ok=False)

def on_encoded(frame_id, gray, jpeg, quality):
    try:
        digest = hashlib.sha256(jpeg).hexdigest() if EXPORT_REFERENCE else None
        psnr = None
        if MEASURE_LOCAL_PSNR:
            restored = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_GRAYSCALE)
            if restored is None or restored.shape != gray.shape:
                raise ValueError("TX JPEG decode failed")
            psnr = float(cv2.PSNR(gray, restored))
        tx_log.append(dict(frame_id=int(frame_id), jpeg_bytes=len(jpeg),
                           quality=int(quality), local_psnr_db=psnr))
        if EXPORT_REFERENCE:
            np.savez_compressed(reference_dir / f"{frame_id}.npz",
                                gray=gray, jpeg_sha256=digest)
    except Exception as exc:
        tx_errors.append(repr(exc))

## 3. Run TX — 相機、壓縮、封包、OFDM、即時顯示

In [ ]:
summary = run_live(ROLE, BITFILE, OUTPUT_ROOT, camera=CAMERA, fps=FPS,
                   amplitude=AMPLITUDE, show_preview=LIVE_PREVIEW,
                   recording_codec=RECORDING_CODEC, jpeg_quality=JPEG_QUALITY,
                   duration_s=DURATION_S, on_encoded=on_encoded)

## 4. TX metrics

In [ ]:
if summary.get("latency_report"):
    show_latency(summary["latency_report"])
print(json.dumps(summary, indent=2, default=str, ensure_ascii=False))
hist = summary.get("frames_per_block_histogram", {})
if hist:
    x = sorted(map(int, hist))
    plt.bar(x, [hist.get(k, hist.get(str(k))) for k in x])
    plt.xlabel("JPEG frames per data block")
    plt.ylabel("Blocks")
    plt.xticks(x)
    plt.show()

rows = list(tx_log)
(OUTPUT_ROOT / f"tx_{PAIR_SESSION}_metrics.json").write_text(
    json.dumps(rows, indent=2), encoding="utf-8")
print("Analysis errors:", list(tx_errors))
if rows:
    ids = [r["frame_id"] for r in rows]
    for field, ylabel in [("jpeg_bytes", "JPEG bytes"), ("quality", "JPEG quality"),
                          ("local_psnr_db", "Local PSNR (dB)")]:
        points = [(r["frame_id"], r[field]) for r in rows if r[field] is not None]
        if points:
            plt.figure(figsize=(8, 3))
            plt.plot(*zip(*points))
            plt.xlabel("Camera frame ID"); plt.ylabel(ylabel); plt.show()

## 5. 注意事項
- duration 計時於硬體初始化完成並啟動 worker 後開始，停止擷取後仍需排空佇列及送 EOS，總時間可能較長。
- JPEG_BUDGET 是目標而非硬限制；最低品質仍超額時，原有 next_batch 按實際大小減少 block 內張數。
- 未新增 PS block 重傳、定期 pilot calibration 或跨板同步。保留原有硬體協議。
- TX 預覽是送出 block 後提交的 JPEG，不表示 RX 每張均成功顯示。